# GPU Utilization Mirage: from demo to real analysis

`nvidia-smi` GPU-Util can read ~100% while the GPU does a tiny fraction of the math it
can do. The honest "useful work" metric is **MFU** (Model FLOPs Utilization) =
achieved FLOP/s divided by the GPU's theoretical peak.

Four parts:
1. **Microbenchmarks** - small workloads that bracket the GPU-Util vs MFU gap.
2. **Roofline** - put them on a roofline so the gap is principled, not just "obvious."
3. **A real training step** - measure MFU of an actual small transformer.
4. **The fixable gap** - a batch-size sweep where MFU moves from low to high by config alone.

### Before you run
1. Runtime > Change runtime type > Hardware accelerator: **GPU** (free T4 is fine).
2. Run top to bottom (later cells reuse earlier results).
3. Parts 3-4 are the substance.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU."
gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)
!nvidia-smi

In [ ]:
# Peak TFLOPs (dense, vendor spec) + memory bandwidth (GB/s). Optimistic ceilings.
SPECS = {
    "T4":   {"fp32": 8.1,  "fp16": 65.0,  "bw_GBs": 320.0},
    "P100": {"fp32": 9.3,  "fp16": 18.7,  "bw_GBs": 732.0},
    "V100": {"fp32": 14.0, "fp16": 112.0, "bw_GBs": 900.0},
    "A100": {"fp32": 19.5, "fp16": 312.0, "bw_GBs": 1555.0},
    "L4":   {"fp32": 30.3, "fp16": 121.0, "bw_GBs": 300.0},
    "L40":  {"fp32": 90.5, "fp16": 181.0, "bw_GBs": 864.0},
}
def get_specs(name):
    for k, v in SPECS.items():
        if k in name: return k, v
    return None, None
spec_key, SPEC = get_specs(gpu_name)
if SPEC is None:
    print(f"!! {gpu_name} unlisted. Fill SPEC manually for MFU/roofline.")
    SPEC = {"fp32": None, "fp16": None, "bw_GBs": None}
else:
    print(f"{spec_key}: FP32={SPEC['fp32']} TF, FP16 tensor={SPEC['fp16']} TF, BW={SPEC['bw_GBs']} GB/s")
PEAKS = {"fp32": SPEC["fp32"], "fp16": SPEC["fp16"]}
# make FP32 actually FP32 so comparisons stay honest
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [ ]:
import subprocess, threading, time, statistics

class GPUUtilSampler:
    """Polls nvidia-smi GPU-Util in a background thread while your code runs."""
    def __init__(self, interval_ms=50):
        self.interval = interval_ms / 1000.0
        self.samples = []
        self._stop = threading.Event()
        self._thread = None
    def _poll(self):
        while not self._stop.is_set():
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=utilization.gpu",
                     "--format=csv,noheader,nounits"], encoding="utf-8")
                self.samples.append(int(out.strip().splitlines()[0]))
            except Exception:
                pass
            time.sleep(self.interval)
    def __enter__(self):
        self.samples = []; self._stop.clear()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start(); return self
    def __exit__(self, *exc):
        self._stop.set()
        if self._thread: self._thread.join(timeout=1.0)
    def stats(self):
        if not self.samples:
            return {"mean": float("nan"), "max": float("nan"), "n": 0}
        return {"mean": statistics.mean(self.samples),
                "max": max(self.samples), "n": len(self.samples)}

## Part 1: Microbenchmarks - brackets for the gap

In [ ]:
def benchmark(name, fn, flops_per_call, dtype, n_iters=50, warmup=10):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    sampler = GPUUtilSampler(50)
    with sampler:
        t0 = time.perf_counter()
        for _ in range(n_iters): fn()
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
    achieved = (flops_per_call * n_iters) / elapsed
    u = sampler.stats(); peak = PEAKS.get(dtype)
    mfu = (100.0 * achieved / (peak * 1e12)) if peak else float("nan")
    return {"workload": name, "dtype": dtype,
            "achieved_TFLOPs": round(achieved/1e12, 2), "MFU_%": round(mfu, 1),
            "gpu_util_mean_%": round(u["mean"], 1), "gpu_util_max_%": u["max"]}

dev = "cuda"
def big_matmul(size, dtype):
    a = torch.randn(size, size, device=dev, dtype=dtype)
    b = torch.randn(size, size, device=dev, dtype=dtype)
    def fn(): torch.matmul(a, b)
    return fn, 2 * size**3
def many_small_matmuls(size, count, dtype):
    pairs = [(torch.randn(size, size, device=dev, dtype=dtype),
              torch.randn(size, size, device=dev, dtype=dtype)) for _ in range(count)]
    def fn():
        for a, b in pairs: torch.matmul(a, b)
    return fn, 2 * (size**3) * count
def memory_bound(numel, dtype):
    x = torch.randn(numel, device=dev, dtype=dtype)
    def fn(): torch.add(torch.mul(x, 1.0001), 0.5)
    return fn, 2 * numel

In [ ]:
dt16, dt32 = torch.float16, torch.float32
results = []
fn, fl = big_matmul(8192, dt16);            results.append(benchmark("big matmul 8192 (fp16)", fn, fl, "fp16", 50))
fn, fl = big_matmul(8192, dt32);            results.append(benchmark("big matmul 8192 (fp32)", fn, fl, "fp32", 30))
fn, fl = many_small_matmuls(128, 512, dt16);results.append(benchmark("512x tiny matmul 128 (fp16)", fn, fl, "fp16", 200))
fn, fl = memory_bound(64_000_000, dt16);    results.append(benchmark("memory-bound elementwise (fp16)", fn, fl, "fp16", 300))
import pandas as pd
pd.DataFrame(results)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
labels=[r["workload"] for r in results]; util=[r["gpu_util_max_%"] for r in results]; mfu=[r["MFU_%"] for r in results]
x=np.arange(len(labels)); w=0.38
fig,ax=plt.subplots(figsize=(11,6))
b1=ax.bar(x-w/2,util,w,label="GPU-Util (max %)"); b2=ax.bar(x+w/2,mfu,w,label="MFU (%)")
ax.set_ylabel("Percent"); ax.set_title(f"GPU-Util vs MFU on {gpu_name}")
ax.set_xticks(x); ax.set_xticklabels(labels,rotation=20,ha="right"); ax.set_ylim(0,110); ax.legend()
ax.bar_label(b1,fmt="%.0f",padding=2); ax.bar_label(b2,fmt="%.0f",padding=2)
plt.tight_layout(); plt.savefig("gpu_util_vs_mfu.png",dpi=150,bbox_inches="tight"); plt.show()

### Part 1 read
On their own the bottom rows are almost true by definition (a memory-bound op has low
FLOP use *because* it is memory-bound). That is exactly why Part 2 puts them on a
roofline: the point is not "look, a gap," it is "the gap lands where the hardware model
predicts, and GPU-Util can't see it."

## Part 2: Roofline - why the gap is principled

Achievable FLOP/s = min(compute_peak, bandwidth x arithmetic_intensity), where
arithmetic intensity (AI) = FLOPs per byte moved. Left of the ridge point = memory-bound
(sloped roof); right = compute-bound (flat roof). For an NxN matmul, AI = N/3 (fp16).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
assert len(results) >= 4 and SPEC["bw_GBs"] and PEAKS["fp16"], "Run Part 1; need specs."
peak_c = PEAKS["fp16"]*1e12; bw = SPEC["bw_GBs"]*1e9; ridge = peak_c/bw
ai = np.logspace(-1, 4, 300); roof = np.minimum(peak_c, bw*ai)
wl = [("big matmul fp16", 8192/3, results[0]["achieved_TFLOPs"]),
      ("tiny matmul fp16", 128/3, results[2]["achieved_TFLOPs"]),
      ("memory-bound fp16", 0.5,  results[3]["achieved_TFLOPs"])]
plt.figure(figsize=(9,6))
plt.loglog(ai, roof/1e12, "k-", lw=2, label="fp16 roofline")
plt.axvline(ridge, ls="--", color="gray", label=f"ridge ~{ridge:.0f} FLOP/byte")
for n,xx,yy in wl:
    plt.scatter([xx],[yy], s=90, zorder=5)
    plt.annotate(f"{n}\n{yy:.2f} TF",(xx,yy),textcoords="offset points",xytext=(8,6),fontsize=9)
plt.xlabel("arithmetic intensity (FLOP/byte)"); plt.ylabel("achieved (TFLOP/s)")
plt.title(f"Roofline - {gpu_name}"); plt.grid(True,which="both",alpha=0.3); plt.legend()
plt.tight_layout(); plt.savefig("roofline.png",dpi=150,bbox_inches="tight"); plt.show()
print(f"ridge ~ {ridge:.0f} FLOP/byte. Left of it = memory-bound.")

### Part 2 read
The memory-bound op sits far left, pinned near the bandwidth roof - its low MFU is the
*most* it can do at that arithmetic intensity, not a bug. The big matmul sits right of
the ridge (compute-bound), below the flat roof by kernel efficiency and Colab throttling.
Honest caveat: the simple roofline ignores kernel-launch overhead and occupancy, which
is why tiny matmuls fall below even the memory roof.

## Part 3: A real training step (not a synthetic matmul)

One step of a small GPT (attention + MLP + backward + optimizer). MFU here is computed
with the standard nanoGPT/PaLM formula: flops_per_token = 6*params + 12*L*H*head_dim*seq.
The 6*params term counts only params that do matmuls, so token/position embeddings
(lookups) are excluded - including them would inflate MFU.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd
        self.qkv=nn.Linear(n_embd,3*n_embd); self.proj=nn.Linear(n_embd,n_embd)
    def forward(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)
class Block(nn.Module):
    def __init__(self,n_embd,n_head):
        super().__init__()
        self.ln1=nn.LayerNorm(n_embd); self.attn=CausalSelfAttention(n_embd,n_head)
        self.ln2=nn.LayerNorm(n_embd)
        self.mlp=nn.Sequential(nn.Linear(n_embd,4*n_embd),nn.GELU(),nn.Linear(4*n_embd,n_embd))
    def forward(self,x):
        x=x+self.attn(self.ln1(x)); x=x+self.mlp(self.ln2(x)); return x
class MiniGPT(nn.Module):
    def __init__(self,vocab,n_embd,n_head,n_layer,block_size):
        super().__init__(); self.block_size=block_size
        self.tok=nn.Embedding(vocab,n_embd); self.pos=nn.Embedding(block_size,n_embd)
        self.blocks=nn.ModuleList([Block(n_embd,n_head) for _ in range(n_layer)])
        self.lnf=nn.LayerNorm(n_embd); self.head=nn.Linear(n_embd,vocab,bias=False)
        self.cfg=dict(vocab=vocab,n_embd=n_embd,n_head=n_head,n_layer=n_layer,block_size=block_size)
    def forward(self,idx,targets=None):
        B,T=idx.shape; pos=torch.arange(T,device=idx.device)
        x=self.tok(idx)+self.pos(pos)[None,:,:]
        for blk in self.blocks: x=blk(x)
        x=self.lnf(x); logits=self.head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1))
        return logits,loss
    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.tok.weight.numel()   # token embedding = lookup, not a matmul
            n -= self.pos.weight.numel()   # position embedding = lookup, not a matmul
        return n

In [ ]:
def gpt_flops_per_step(model, batch, seq):
    N=model.num_params(non_embedding=True); c=model.cfg
    L,H=c["n_layer"],c["n_head"]; Q=c["n_embd"]//c["n_head"]; T=seq
    flops_per_token = 6*N + 12*L*H*Q*T
    return flops_per_token * T * batch   # fwd+bwd, whole batch

def measure_training_mfu(model, batch, seq, dtype="fp16", n_iters=20, warmup=5):
    vocab=model.cfg["vocab"]
    x=torch.randint(0,vocab,(batch,seq),device="cuda")
    y=torch.randint(0,vocab,(batch,seq),device="cuda")
    opt=torch.optim.AdamW(model.parameters(),lr=3e-4)
    amp_dtype = torch.float16 if dtype=="fp16" else (torch.bfloat16 if dtype=="bf16" else None)
    use_amp = amp_dtype is not None
    try: scaler=torch.amp.GradScaler("cuda",enabled=(amp_dtype==torch.float16))
    except TypeError: scaler=torch.amp.GradScaler(enabled=(amp_dtype==torch.float16))
    except AttributeError: scaler=torch.cuda.amp.GradScaler(enabled=(amp_dtype==torch.float16))
    def step():
        opt.zero_grad(set_to_none=True)
        if use_amp:
            with torch.autocast("cuda",dtype=amp_dtype):
                _,loss=model(x,y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            _,loss=model(x,y); loss.backward(); opt.step()
    for _ in range(warmup): step()
    torch.cuda.synchronize()
    s=GPUUtilSampler(50)
    with s:
        t0=time.perf_counter()
        for _ in range(n_iters): step()
        torch.cuda.synchronize(); dt=(time.perf_counter()-t0)/n_iters
    achieved=gpt_flops_per_step(model,batch,seq)/dt
    peak=PEAKS["fp16"] if dtype in ("fp16","bf16") else PEAKS["fp32"]
    mfu=100*achieved/(peak*1e12) if peak else float("nan")
    u=s.stats()
    return {"batch":batch,"seq":seq,"dtype":dtype,"TFLOPs":round(achieved/1e12,2),
            "MFU_%":round(mfu,1),"util_max_%":u["max"],"ms_per_step":round(dt*1000,1)}

In [ ]:
def make_model():
    return MiniGPT(vocab=8192, n_embd=384, n_head=6, n_layer=6, block_size=256)
SEQ = 256
m = make_model().cuda()
print(f"model params (total): {m.num_params()/1e6:.1f}M | non-embedding (used for FLOPs): {m.num_params(non_embedding=True)/1e6:.1f}M")
print("(vocab is synthetic; we measure throughput, not learning)")
single = measure_training_mfu(m, batch=8, seq=SEQ, dtype="fp16", n_iters=20, warmup=5)
del m; torch.cuda.empty_cache()
print(single)

## Part 4: The fixable gap - batch-size sweep

Same model, same GPU, only batch changes. Small batches starve the GPU (low MFU); larger
batches amortize overhead and feed the tensor cores (higher MFU) until a plateau or OOM.
This is the actionable finding: free MFU from config.

In [ ]:
def batch_sweep(make_model, batches, seq, dtype="fp16"):
    rows=[]
    for b in batches:
        torch.cuda.empty_cache()
        try:
            model=make_model().cuda()
            r=measure_training_mfu(model,b,seq,dtype=dtype,n_iters=15,warmup=4)
            rows.append(r); print(r); del model
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"batch={b}: OOM, stopping sweep"); torch.cuda.empty_cache(); break
            raise
    return rows

sweep = batch_sweep(make_model, [1,2,4,8,16,32,48], SEQ, dtype="fp16")
import pandas as pd, matplotlib.pyplot as plt
print(pd.DataFrame(sweep))
bs=[r["batch"] for r in sweep]; mf=[r["MFU_%"] for r in sweep]; ut=[r["util_max_%"] for r in sweep]
fig,ax=plt.subplots(figsize=(9,6))
ax.plot(bs,mf,"o-",lw=2,label="MFU %"); ax.plot(bs,ut,"s--",lw=1.5,label="GPU-Util max %")
ax.set_xlabel("batch size"); ax.set_ylabel("percent"); ax.set_ylim(0,105)
ax.set_title(f"Real GPT training: MFU vs batch size - {gpu_name}")
for xx,yy in zip(bs,mf): ax.annotate(f"{yy:.0f}",(xx,yy),textcoords="offset points",xytext=(0,8),fontsize=9)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("mfu_vs_batch.png",dpi=150,bbox_inches="tight"); plt.show()
if len(sweep)>=2:
    lo=sweep[0]; hi=max(sweep,key=lambda r:r["MFU_%"])
    print(f"batch={lo['batch']} -> {lo['MFU_%']}% MFU; batch={hi['batch']} -> {hi['MFU_%']}% MFU "
          f"({hi['MFU_%']/max(lo['MFU_%'],0.1):.1f}x) with zero hardware change.")
    # also report a realistic-batch delta (avoids leaning on the batch=1 floor)
    real = [r for r in sweep if r["batch"] >= 8]
    if real:
        rb = real[0]
        print(f"realistic baseline batch={rb['batch']} -> {rb['MFU_%']}% MFU; "
              f"best -> {hi['MFU_%']}% ({hi['MFU_%']/max(rb['MFU_%'],0.1):.1f}x).")

In [ ]:
# Precision lever: same model/batch, fp32 vs fp16 autocast.
prec=[]
for d in ["fp32","fp16"]:
    torch.cuda.empty_cache(); mm=make_model().cuda()
    try: prec.append(measure_training_mfu(mm,batch=8,seq=SEQ,dtype=d,n_iters=15,warmup=4))
    except RuntimeError as e: print(d,"failed:",e)
    del mm
import pandas as pd
print(pd.DataFrame(prec))

In [ ]:
# Cost framing (honest: vs theoretical peak, not a literal refund)
T4_RATE = 0.35   # $/hr on-demand; edit for your provider
ref = max(sweep, key=lambda r: r["MFU_%"]) if sweep else single
mfu = ref["MFU_%"]/100.0
print(f"Best real-training MFU here: {ref['MFU_%']}% (batch={ref['batch']}).")
print(f"That uses ~{mfu:.0%} of the T4 compute peak.")
print(f"Caveat: 100% MFU is unreachable, so treat the rest as headroom-vs-peak.")
print(f"Rough headroom at ${T4_RATE}/hr: ~${T4_RATE*(1-mfu):.2f}/hr of GPU-time not on model FLOPs.")

## The honest story for the post

Now you have real, non-obvious numbers:
- a roofline that *explains* the microbenchmark gap (roofline.png),
- a real GPT training step at a real MFU,
- MFU vs batch size showing the fixable jump (mfu_vs_batch.png).

Draft (fill in YOUR measured numbers; prefer the realistic-batch delta over the batch=1 ratio):
> On a T4, a real GPT training step ran at only ~X% MFU - the GPU looked busy (Y% util)
> but was doing a fraction of its possible math. Raising batch size from N1 to N2 took it
> to ~Z% MFU, no hardware change. GPU-Util read "busy" the whole time. The dashboard
> metric and the useful-work metric are not the same thing.

Attach mfu_vs_batch.png. Report only what you measured on this run.